# FlashEats — Class 5 Investigation
**Mission:** assemble trustworthy evidence about late deliveries.

Rules: no ML for first 90 minutes; preserve raw API responses; state your definition of late; never silently drop failures.

In [1]:
# [replaced classroom cell] The Colab/zip loader and `!pip install` are not needed here:
# the client pack lives in source_systems/ and dependencies come from requirements.txt.
import os, sys, json, time, sqlite3, subprocess, zipfile
from pathlib import Path
import pandas as pd
import requests
import matplotlib.pyplot as plt

ROOT = Path.cwd()
while not (ROOT / "source_systems").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
BASE = ROOT / "source_systems"                    # read-only client systems
sys.path.insert(0, str(ROOT / "src"))
OUT = ROOT / "Challenges" / "output"             # scratch outputs of the challenge notebooks
OUT.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_columns", 100); pd.set_option("display.width", 170); pd.set_option("display.max_colwidth", 90)
print("BASE:", BASE, "| database exists:", (BASE / "database" / "flasheats.db").exists())

BASE: B:\Trimester-9\FDE\assignment-2\source_systems | database exists: True


## Challenge 1 — How large is the late-delivery problem?
Write down your definition of late, denominator, cancellation policy, and missing-timestamp policy before querying.

> ✅ **Answer**

Decided **before** querying:

| Decision | Choice | Why |
|---|---|---|
| Late means | `actual_delivery_at − promised_eta > 0 min` | the promise the customer saw is the yardstick |
| Denominator | delivered orders with both timestamps, **one row per order** | the raw table repeats 3 orders |
| Cancellations | excluded | nothing was delivered, so nothing can be late |
| Missing actual delivery time | excluded **and counted** (unknown outcome, not on time) | 37 orders; bounds the KPI |
| Grain | one row per `order_id` after de-duplication | `COUNT(*)` ≠ `COUNT(DISTINCT order_id)` |

This notebook answers the Starter challenges **in SQL first** (the Student notebook does them in pandas), to show that the same definitions give the same numbers in both tools.

In [2]:
con=sqlite3.connect(BASE/'database'/'flasheats.db')
pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'",con)

,name
0,customers
1,drivers
2,restaurants
3,orders


In [3]:
query='''
-- YOUR SQL HERE
SELECT * FROM orders LIMIT 5;
'''
pd.read_sql(query,con)

,order_id,customer_id,restaurant_id,driver_id,city,created_at,promised_eta,pickup_at,actual_delivery_at,final_status,distance_km_estimate,traffic_bucket,weather_bucket
0,O00001,C0168,R009,D103,Bengaluru,2026-08-13T12:56:00,2026-08-13T14:11:36,2026-08-13T13:35:49.825561,2026-08-13T14:22:25.035899,delivered,18.00,medium,clear
1,O00002,C0043,R018,D083,Bengaluru,2026-08-01T18:36:00,2026-08-01T19:43:16.627988,2026-08-01T18:58:24.735336,2026-08-01T19:47:14.818533,delivered,9.37,severe,clear
2,O00003,C0855,R010,D039,Bengaluru,2026-08-01T19:35:00,2026-08-01T20:17:52.941967,2026-08-01T19:57:50.820366,2026-08-01T20:13:28.061191,delivered,2.42,medium,clear
3,O00004,C0229,R030,D038,Bengaluru,2026-08-09T18:16:00,2026-08-09T19:30:19.067006,2026-08-09T18:46:48.725485,NaN,cancelled,14.20,high,rain
4,O00005,C0040,R004,D047,Bengaluru,2026-08-06T20:35:00,2026-08-06T21:54:48,2026-08-06T21:18:38.024280,2026-08-06T22:15:49.290713,delivered,18.00,high,clear


In [4]:
# Three SQL versions of the same KPI - each fixes one trap, so the difference is explainable, not mysterious
POP_SQL = """
WITH one_row AS (SELECT o.* FROM orders o JOIN (SELECT MIN(rowid) rid FROM orders GROUP BY order_id) f ON f.rid = o.rowid),
pop AS (SELECT *, (julianday(actual_delivery_at) - julianday(promised_eta)) * 1440 AS delay_min
        FROM one_row WHERE lower(trim(final_status)) = 'delivered' AND actual_delivery_at IS NOT NULL
          AND promised_eta >= created_at AND actual_delivery_at >= pickup_at)
"""   # the validated population, reused by the next challenges
naive = pd.read_sql("""
SELECT COUNT(*) AS delivered, SUM(julianday(actual_delivery_at) > julianday(promised_eta)) AS late
FROM orders WHERE final_status = 'delivered' AND actual_delivery_at IS NOT NULL""", con)

dedup = pd.read_sql("""
WITH one_row AS (SELECT o.* FROM orders o JOIN (SELECT MIN(rowid) rid FROM orders GROUP BY order_id) f ON f.rid = o.rowid)
SELECT COUNT(*) AS delivered, SUM(julianday(actual_delivery_at) > julianday(promised_eta)) AS late
FROM one_row WHERE lower(trim(final_status)) = 'delivered' AND actual_delivery_at IS NOT NULL""", con)

validated = pd.read_sql((ROOT / "sql" / "independent_late_rate_check.sql").read_text(encoding="utf-8"), con)   # + chronology rules

kpi = pd.DataFrame({
    "version": ["naive: status = 'delivered'", "+ one row per order, 'Delivered' = 'delivered'", "+ chronology rules (promise after creation, delivery after pickup)"],
    "delivered": [int(naive.delivered[0]), int(dedup.delivered[0]), int(validated.valid_delivered[0])],
    "late": [int(naive.late[0]), int(dedup.late[0]), int(validated.late_orders[0])]})
kpi["late_rate_%"] = (100 * kpi["late"] / kpi["delivered"]).round(2)
display(kpi)

print(pd.read_sql("SELECT COUNT(*) rows_, COUNT(DISTINCT order_id) orders_ FROM orders", con).to_string(index=False))
print(pd.read_sql("SELECT final_status, COUNT(*) n FROM orders GROUP BY final_status", con).to_string(index=False))
print("delivered but no actual_delivery_at:", pd.read_sql("SELECT COUNT(DISTINCT order_id) n FROM orders WHERE lower(final_status)='delivered' AND actual_delivery_at IS NULL", con).n[0])

,version,delivered,late,late_rate_%
0,naive: status = 'delivered',1493,841,56.33
1,"+ one row per order, 'Delivered' = 'delivered'",1495,843,56.39
2,"+ chronology rules (promise after creation, delivery after pickup)",1486,837,56.33


 rows_  orders_
  1603     1600
final_status    n
   Delivered    5
   cancelled   68
   delivered 1530
delivered but no actual_delivery_at: 37


> ✅ **Answer**

- The **naive** query gives 1493 / 841: it misses the 5 `Delivered` rows (capital D) and double-counts the 3 duplicated orders.
- **One row per order** plus status normalisation gives **1495 / 843 = 56.39 %**. This matches the class answer and the leadership claim.
- Applying the **chronology rules** gives **1486 / 837 = 56.33 %**. Four promises were made before the order existed and five deliveries happened before pickup, so their lateness is meaningless. The pipeline publishes this figure.
- 37 delivered orders have no delivery time, so the true rate lies between 55.0 % (all on time) and 57.4 % (all late).

**Checkpoint:** when two teams disagree, the cause is almost always one of these three traps, not arithmetic.

### Checkpoint
Compare counts with another team. If they differ, investigate row grain, duplicates, cancellations, missing timestamps, and definitions.

## Challenge 2 — Operations says traffic is the cause
Test the claim. Treat traffic/weather/distance as descriptive signals, not causal proof.

In [5]:
# Your analysis here

In [6]:
q = POP_SQL + """
SELECT {dim} AS bucket, COUNT(*) AS orders, ROUND(100.0 * AVG(delay_min > 0), 1) AS late_rate_pct, ROUND(AVG(delay_min), 1) AS avg_delay_min
FROM pop GROUP BY 1 ORDER BY 1"""
for name, dim in [("traffic", "lower(trim(traffic_bucket))"), ("weather", "weather_bucket"),
                  ("distance band", "CASE WHEN distance_km_estimate <= 6 THEN '0-6 km' WHEN distance_km_estimate <= 12 THEN '6-12 km' ELSE '12+ km' END"),
                  ("hour of day", "CAST(strftime('%H', created_at) AS INTEGER)")]:
    print(f"--- {name}"); display(pd.read_sql(q.format(dim=dim), con))

--- traffic


,bucket,orders,late_rate_pct,avg_delay_min
0,high,421,66.5,5.2
1,low,358,49.4,0.9
2,medium,585,51.1,1.8
3,severe,122,66.4,6.4


--- weather


,bucket,orders,late_rate_pct,avg_delay_min
0,clear,1185,53.2,2.0
1,heavy_rain,72,73.6,7.7
2,rain,229,66.8,6.1


--- distance band


,bucket,orders,late_rate_pct,avg_delay_min
0,0-6 km,96,51.0,1.1
1,12+ km,1084,57.0,3.2
2,6-12 km,306,55.6,2.6


--- hour of day


,bucket,orders,late_rate_pct,avg_delay_min
0,11,94,59.6,3.2
1,12,123,55.3,2.5
2,13,93,53.8,2.2
3,14,73,54.8,2.2
4,15,56,67.9,4.7
5,16,75,60.0,4.1
6,17,127,57.5,4.2
7,18,222,62.2,4.6
8,19,204,53.4,3.0
9,20,171,53.2,1.7


> ✅ **Answer**

**Traffic is associated with lateness.** The late rate rises from 49 % in low traffic to 66 % in high or severe traffic. Distance shows only a weak gradient, from 51 % under 6 km to 57 % above 12 km, and the hour of day moves the rate between 43 % and 72 %.

**Why this is not proof:**
- The traffic bucket is a label attached to the order, not a measurement taken on the route.
- The evening peak raises traffic, kitchen load and driver scarcity at the same time.
- A later finding in `insights/` shows that almost all lateness builds up *before pickup*. The road is not where the time is lost, even when traffic is high.

## Challenge 3 — Customer Support disagrees
Inspect support tickets. What are customers actually complaining about? Are ticket records unique?

In [7]:
tickets=pd.read_csv(BASE/'data'/'support_tickets.csv'); tickets.head()

,ticket_id,order_id,created_at,category,customer_message
0,T00001,NaN,2026-08-26T23:34:00,late_delivery,My order is already past the promised time.
1,T00002,NaN,2026-08-24T13:49:00,eta_changed,The ETA keeps changing and the food is still not here.
2,T00003,NaN,2026-08-25T18:22:00,status_mismatch,The app says picking up but the restaurant says it is ready.
3,T00004,O00037,2026-08-09T21:09:00,Late Delivery,The rider has not moved for 15 minutes.
4,T00005,O00040,2026-08-11T20:15:00,late_delivery,My order is already past the promised time.


In [8]:
print("rows:", len(tickets), "| unique ticket ids:", tickets.ticket_id.nunique(), "| exact duplicate rows:", tickets.duplicated().sum(),
      "| tickets without order_id:", tickets.order_id.isna().sum())
print("raw categories:", sorted(tickets.category.unique()))
t = tickets.drop_duplicates().assign(category_norm=lambda d: d.category.str.strip().str.lower().str.replace(r"[\s-]+", "_", regex=True))
display(t.category_norm.value_counts().rename("tickets").to_frame())
# does the customer story match the operational story? (join tickets to the validated KPI population)
pop = pd.read_sql(POP_SQL + " SELECT order_id, delay_min FROM pop", con)
v = t.merge(pop, on="order_id", how="left")
display(v.groupby("category_norm").agg(tickets=("ticket_id", "size"), matched=("delay_min", "count"),
                                        late_share_pct=("delay_min", lambda s: round(100 * (s > 0).mean(), 1)), median_delay=("delay_min", "median")).round(1))

rows: 202 | unique ticket ids: 201 | exact duplicate rows: 1 | tickets without order_id: 3
raw categories: ['ETA issue', 'Late Delivery', 'driver_not_moving', 'eta_changed', 'late_delivery', 'late_delivery ', 'ready_but_waiting', 'restaurant_delay', 'status_mismatch']


,tickets
category_norm,
late_delivery,39
eta_changed,37
restaurant_delay,37
ready_but_waiting,33
status_mismatch,29
driver_not_moving,25
eta_issue,1


,tickets,matched,late_share_pct,median_delay
category_norm,,,,
driver_not_moving,25,25,68.0,10.4
eta_changed,37,35,75.7,7.7
eta_issue,1,1,100.0,5.4
late_delivery,39,38,84.6,14.8
ready_but_waiting,33,33,90.9,12.2
restaurant_delay,37,37,75.7,10.5
status_mismatch,29,28,89.7,12.7


> ✅ **Answer**

- **Customer view:** complaints are about lateness, the ETA changing and the restaurant. `ETA issue` is kept separate from `eta_changed`, because merging them is a meaning decision for the Support Lead.
- **System view:** 68–91 % of ticketed orders really were late, depending on the category, with median delays of 8–15 minutes. Customers are not crying wolf.
- **Ticket IDs are not unique.** One ticket appears twice as an exact duplicate, and three tickets have no order ID at all.
- **What tickets add:** the customer's belief about the cause and the moment they lost patience. No timestamp captures either.

## Challenge 4 — Retrieve Dispatch data reliably

In [9]:
# [classroom cell, made robust] start the mock Dispatch API and wait until it is healthy (instead of a fixed sleep)
api_script = BASE / "api" / "mock_dispatch_api.py"
api_proc = subprocess.Popen([sys.executable, str(api_script)], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(60):
    try:
        if requests.get("http://127.0.0.1:8000/health", timeout=2).ok:
            break
    except requests.RequestException:
        time.sleep(0.25)
health = requests.get("http://127.0.0.1:8000/health", timeout=10)
print(health.status_code, health.json())

200 {'service': 'flasheats-dispatch-api', 'status': 'ok'}


In [10]:
url='http://127.0.0.1:8000/dispatch/orders'
r=requests.get(url,params={'page':1,'page_size':50},timeout=10)
print(r.status_code); payload=r.json(); print(payload.keys(), len(payload.get('data',[])), payload.get('has_more'))

200
dict_keys(['data', 'has_more', 'page', 'page_size', 'total_records']) 50 True


### Your task
Implement pagination + retry + raw-page preservation. Refuse to claim success if ingestion is incomplete.

In [11]:
RAW_SUBDIR = "raw_dispatch_starter"

# [classroom skeleton completed] reliable paginated ingestion
# Requirements from the skeleton: fetch every page, retry HTTP 500 / 429, preserve successful raw
# responses, fail clearly on unrecoverable errors, verify the final record count.
API_URL = "http://127.0.0.1:8000/dispatch/orders"
RAW_DIR = OUT / RAW_SUBDIR                           # NOT inside source_systems/ - the client snapshot stays untouched
RAW_DIR.mkdir(parents=True, exist_ok=True)

def fetch_all_dispatch_orders(page_size=100, max_retries=4, timeout=10):
    records, seen, page, expected_total, log = [], set(), 1, None, []
    while True:
        for attempt in range(1, max_retries + 1):
            try:
                r = requests.get(API_URL, params={"page": page, "page_size": page_size}, timeout=timeout)
            except (requests.Timeout, requests.ConnectionError) as exc:
                log.append((page, attempt, type(exc).__name__)); time.sleep(min(2 ** (attempt - 1), 4)); continue
            if r.status_code in (429, 500, 502, 503, 504):
                wait = float(r.json().get("retry_after_seconds", 0)) if r.status_code == 429 else min(2 ** (attempt - 1), 4)
                log.append((page, attempt, r.status_code)); time.sleep(wait); continue
            r.raise_for_status()                          # 4xx = configuration bug, never retried
            payload = r.json()
            break
        else:
            raise RuntimeError(f"page {page} failed after {max_retries} attempts: {log[-3:]}")
        if not isinstance(payload.get("data"), list):     # HTTP 200 is not proof: validate the body
            raise RuntimeError(f"page {page}: unexpected payload keys {list(payload)}")
        (RAW_DIR / f"dispatch_page_{page:03d}.json").write_text(json.dumps(payload, indent=1), encoding="utf-8")
        expected_total = payload.get("total_records", expected_total)
        for rec in payload["data"]:
            if rec["order_id"] in seen:
                raise RuntimeError(f"order {rec['order_id']} returned twice - pagination overlap")
            seen.add(rec["order_id"]); records.append(rec)
        if not payload.get("has_more"):
            break
        page += 1
    if expected_total is not None and len(records) != expected_total:
        raise RuntimeError(f"incomplete ingestion: got {len(records)}, server reports {expected_total}")
    return records, log

dispatch_records, retry_log = fetch_all_dispatch_orders()
dispatch_df = pd.DataFrame(dispatch_records)
print("pages saved:", len(list(RAW_DIR.glob("dispatch_page_*.json"))), "| retried:", retry_log)
print("records:", len(dispatch_df), "| unique orders:", dispatch_df["order_id"].nunique(), "| reassigned:", dispatch_df["reassigned_at"].notna().sum())
display(dispatch_df.head())

pages saved: 16 | retried: [(3, 1, 500), (5, 1, 429)]
records: 1600 | unique orders: 1600 | reassigned: 95


,assigned_at,current_delivery_eta,dispatch_status,driver_id,estimated_pickup_at,eta_model_version,order_id,original_driver_id,reassigned_at
0,2026-08-13T13:01:09.659644,2026-08-13T14:19:15.546424,completed,D103,2026-08-13T13:14:00,eta-v3.2,O00001,D103,NaN
1,2026-08-01T18:37:09.708219,2026-08-01T20:01:16.627988,completed,D083,2026-08-01T18:54:00,eta-v3.2,O00002,D083,NaN
2,2026-08-01T19:35:00.002443,2026-08-01T20:22:59.204700,completed,D039,2026-08-01T19:53:00,eta-v3.2,O00003,D039,NaN
3,2026-08-09T18:17:16.087835,2026-08-09T19:51:58.608301,cancelled,D038,2026-08-09T18:34:00,eta-v3.2,O00004,D038,NaN
4,2026-08-06T20:36:55.346822,2026-08-06T22:13:17.537116,completed,D047,2026-08-06T20:53:00,eta-v3.1,O00005,D047,NaN


## Challenge 5 — Driver events
What is observed vs inferred? Can you find a reliable driver-arrival-at-restaurant event?

In [12]:
with open(BASE/'data'/'driver_events.json') as f: driver_events=json.load(f)
driver_events[0]

{'driver_id': 'D001',
 'events': [{'order_id': 'O00162',
   'type': 'assigned',
   'timestamp': '2026-08-26T17:53:25.977803'},
  {'order_id': 'O00162',
   'type': 'gps_ping',
   'timestamp': '2026-08-26T18:10:59.372420',
   'lat': 12.826339,
   'lon': 77.448282},
  {'order_id': 'O00162',
   'type': 'gps_ping',
   'timestamp': '2026-08-26T18:28:32.767037',
   'lat': 12.876013,
   'lon': 77.477707},
  {'order_id': 'O00162',
   'type': 'gps_ping',
   'timestamp': '2026-08-26T18:46:06.161653',
   'lat': 12.925686,
   'lon': 77.507133},
  {'order_id': 'O00162',
   'type': 'gps_ping',
   'timestamp': '2026-08-26T19:03:39.556270',
   'lat': 12.97536,
   'lon': 77.536558},
  {'order_id': 'O00162',
   'type': 'picked_up',
   'timestamp': '2026-08-26T18:27:32.131791'},
  {'order_id': 'O00162',
   'type': 'delivered',
   'timestamp': '2026-08-26T19:21:12.950887'},
  {'order_id': 'O00180',
   'type': 'assigned',
   'timestamp': '2026-08-08T18:05:08.439625'},
  {'order_id': 'O00180',
   'type': 'gp

In [13]:
ev = pd.DataFrame([{"driver_id": d["driver_id"], **e} for d in driver_events for e in d["events"]])
ev["timestamp"] = pd.to_datetime(ev["timestamp"], format="mixed")
display(ev.type.value_counts())
print("explicit arrived-at-restaurant events:", (ev.type == "arrived_at_restaurant").sum())

# Beyond the class: can ARRIVAL be inferred from GPS (a geofence around the restaurant)?
import numpy as np
orders = pd.read_sql("SELECT DISTINCT order_id, restaurant_id, pickup_at FROM orders", con).drop_duplicates("order_id")
orders["pickup_at"] = pd.to_datetime(orders.pickup_at, format="mixed")
rest = pd.read_sql("SELECT restaurant_id, lat r_lat, lon r_lon FROM restaurants WHERE lat BETWEEN 12.5 AND 13.5 AND lon BETWEEN 77.2 AND 78", con)
p = ev[ev.type == "gps_ping"].merge(orders, on="order_id").merge(rest, on="restaurant_id")
p = p[p.timestamp <= p.pickup_at].sort_values("timestamp")
R = np.radians
p["km"] = 6371 * 2 * np.arcsin(np.sqrt(np.sin((R(p.lat) - R(p.r_lat)) / 2) ** 2 + np.cos(R(p.lat)) * np.cos(R(p.r_lat)) * np.sin((R(p.lon) - R(p.r_lon)) / 2) ** 2))
g = p.groupby("order_id").km.agg(first="first", last="last", n="size")
g = g[g.n >= 2]
print(f"orders with >=2 pre-pickup pings: {len(g)} | last ping closer to the restaurant than the first: {(g['last'] < g['first']).mean():.1%} "
      f"| FARTHER: {(g['last'] > g['first']).mean():.1%} | median last-ping distance: {g['last'].median():.1f} km")

# ...and do the pings look MEASURED or DRAWN? A real device track wiggles and has uneven gaps;
# an app that interpolates a straight line between two points does not.
allp = ev[ev.type == "gps_ping"].merge(orders, on="order_id")
print(f"pings before pickup: {(allp.timestamp <= allp.pickup_at).mean():.1%} of {len(allp)} (Session 6 expected pings only before pickup)")
def straightness(t):
    if len(t) < 3:
        return np.nan
    x = (t.timestamp - t.timestamp.iloc[0]).dt.total_seconds().to_numpy(float)
    return min(1.0 if np.ptp(t[c]) == 0 else np.corrcoef(x, t[c].to_numpy(float))[0, 1] ** 2 for c in ("lat", "lon"))
inside = ev[(ev.type == "gps_ping") & ev.lat.between(12.5, 13.5) & ev.lon.between(77.2, 78.0)].sort_values("timestamp")
shape = inside.groupby("order_id")[["timestamp", "lat", "lon"]].apply(straightness).dropna()
print(f"tracks with >=3 in-area pings: {len(shape)} | perfectly straight, constant-speed (R^2 >= 0.999): {(shape >= 0.999).mean():.1%}")

type
gps_ping     5371
assigned     1600
picked_up    1532
delivered    1532
Name: count, dtype: int64

explicit arrived-at-restaurant events: 0


orders with >=2 pre-pickup pings: 374 | last ping closer to the restaurant than the first: 0.8% | FARTHER: 99.2% | median last-ping distance: 4.6 km
pings before pickup: 30.6% of 5371 (Session 6 expected pings only before pickup)


tracks with >=3 in-area pings: 1111 | perfectly straight, constant-speed (R^2 >= 0.999): 91.3%


> ✅ **Answer**

| Event / fact | Observed? | Source | Reliability concern |
|---|---|---|---|
| Driver assigned | **Observed** | driver app `assigned`, Dispatch `assigned_at` | 14 orders show `picked_up` *before* `assigned` |
| Driver at restaurant | **Not observed; not inferable either** | GPS pings | before pickup the pings move *away* from the restaurant in about 99 % of orders, and about 9 in 10 tracks are perfect straight constant-speed lines. The pings look drawn by the app, not measured, so a geofence would invent arrivals |
| Picked up | **Observed** | `orders.pickup_at`, driver app | agree to the second for 1527 orders |
| Delivered | **Observed** | `orders.actual_delivery_at`, driver app | 37 delivered orders lack the DB timestamp but have the driver event |

The class exercise names GPS as the signal from which arrival could be *inferred*. On this feed it cannot: the pings look interpolated. Session 6 also said pings stop at pickup; here about 30 % fall before pickup and the rest after it. The only fix is a new event, such as a driver-app tap on arrival, and a question to Fleet Ops: are these pings device readings?

## Final FDE recommendation
Prepare one slide: problem size, evidence, uncertainty, missing instrumentation, next step.

**Would you build the AI delay predictor now? Why or why not?**

> ✅ **Answer**

**Problem size:** 56.3 % of validated deliveries are late (837 of 1486). The median late order is 8 minutes late, and one in four deliveries is more than 10 minutes late.

**Evidence:** lateness is associated with traffic, time of day and the weather label. The weather label turns out not to match observed weather (see Class 6 and `insights/`).

**Uncertainty:**
- 37 orders have an unknown outcome.
- 9 orders have contradictory timestamps.
- No arrival event exists.
- Four definitions of "late" compete, and none has an owner.

**Missing instrumentation:** an *arrived at restaurant* tap, a mandatory delivery timestamp, reassignment written back to the orders table, and a cancellation timestamp.

**Build the AI delay predictor now? No.** It would learn from promises that are sometimes made before the order exists, and it would lack the one event that separates kitchen delay from rider delay. First ship the rule-based pickup-overrun trigger from the pipeline's decision layer, which is 96 % precise on the held-out week. That trigger becomes the baseline any future model has to beat.

In [14]:
# cleanup: close the database and stop the mock API started above
try:
    con.close()
except Exception:
    pass
try:
    api_proc.terminate(); api_proc.wait(timeout=5)
except Exception:
    pass
print("cleanup complete")

cleanup complete
